In [1]:
# Cell A — All Models Registry

# ── make sure config constants are available ──────────────────────────
try:
    from shared.config import *          # loads RANDOM_STATE, N_TRIALS, PSO_TOP_N, etc.
except ImportError:
    pass  # already imported in Cell 0

# fallback defaults in case shared.config is missing or not yet run
if "RANDOM_STATE" not in dir():
    RANDOM_STATE = 42
if "N_TRIALS" not in dir():
    N_TRIALS = 50
if "PSO_TOP_N" not in dir():
    PSO_TOP_N = 20
if "PSO_PARTICLES" not in dir():
    PSO_PARTICLES = 10
if "PSO_ITERS" not in dir():
    PSO_ITERS = 30
if "SAMPLE_PSO" not in dir():
    SAMPLE_PSO = 50_000
if "SAMPLE_SMOTE" not in dir():
    SAMPLE_SMOTE = 100_000

# ── imports ───────────────────────────────────────────────────────────
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

MODELS = {
    "Decision Tree":  DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest":  RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE),
    "XGBoost":        XGBClassifier(tree_method="hist", device="cpu", random_state=RANDOM_STATE, verbosity=0),
    "KNN":            KNeighborsClassifier(n_jobs=-1),
    "LightGBM":       LGBMClassifier(random_state=RANDOM_STATE, verbosity=-1),
    "SVM":            SVC(probability=True, random_state=RANDOM_STATE),
    "Logistic Reg.":  LogisticRegression(max_iter=1000, n_jobs=-1, random_state=RANDOM_STATE),
    "Naive Bayes":    GaussianNB(),
}

print(f"RANDOM_STATE = {RANDOM_STATE}")
print(f"N_TRIALS     = {N_TRIALS}")
print(f"PSO_TOP_N    = {PSO_TOP_N}")
print(f"\nRegistered {len(MODELS)} models:")
for name in MODELS:
    print(f"  - {name}")

Config loaded successfully
RANDOM_STATE = 42
N_TRIALS     = 30
PSO_TOP_N    = 30

Registered 8 models:
  - Decision Tree
  - Random Forest
  - XGBoost
  - KNN
  - LightGBM
  - SVM
  - Logistic Reg.
  - Naive Bayes


In [ ]:
# Cell B — Optuna Bayesian HPO for All Models
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.metrics import balanced_accuracy_score

def get_objective(model_name):
    def objective(trial):
        if model_name == "Decision Tree":
            clf = DecisionTreeClassifier(
                max_depth=trial.suggest_int("max_depth", 5, 30),
                min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
                min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
                random_state=RANDOM_STATE
            )
        elif model_name == "Random Forest":
            clf = RandomForestClassifier(
                n_estimators=trial.suggest_int("n_estimators", 50, 300),
                max_depth=trial.suggest_int("max_depth", 5, 30),
                min_samples_split=trial.suggest_int("min_samples_split", 2, 10),
                n_jobs=-1, random_state=RANDOM_STATE
            )
        elif model_name == "XGBoost":
            clf = XGBClassifier(
                n_estimators=trial.suggest_int("n_estimators", 200, 600),
                max_depth=trial.suggest_int("max_depth", 4, 10),
                learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                subsample=trial.suggest_float("subsample", 0.6, 1.0),
                colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
                tree_method="hist", device="cpu", random_state=RANDOM_STATE, verbosity=0
            )
        elif model_name == "KNN":
            clf = KNeighborsClassifier(
                n_neighbors=trial.suggest_int("n_neighbors", 3, 20),
                weights=trial.suggest_categorical("weights", ["uniform", "distance"]),
                p=trial.suggest_int("p", 1, 2),
                n_jobs=-1
            )
        elif model_name == "LightGBM":
            clf = LGBMClassifier(
                n_estimators=trial.suggest_int("n_estimators", 100, 500),
                max_depth=trial.suggest_int("max_depth", 4, 12),
                learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                num_leaves=trial.suggest_int("num_leaves", 20, 100),
                random_state=RANDOM_STATE, verbosity=-1
            )
        elif model_name == "SVM":
            clf = SVC(
                C=trial.suggest_float("C", 0.01, 100, log=True),
                kernel=trial.suggest_categorical("kernel", ["rbf", "linear"]),
                gamma=trial.suggest_categorical("gamma", ["scale", "auto"]),
                probability=True, random_state=RANDOM_STATE
            )
        elif model_name == "Logistic Reg.":
            clf = LogisticRegression(
                C=trial.suggest_float("C", 0.01, 100, log=True),
                solver=trial.suggest_categorical("solver", ["lbfgs", "saga"]),
                max_iter=1000, n_jobs=-1, random_state=RANDOM_STATE
            )
        elif model_name == "Naive Bayes":
            clf = GaussianNB(
                var_smoothing=trial.suggest_float("var_smoothing", 1e-10, 1e-7, log=True)
            )
        
        clf.fit(X_train, y_train)
        proba = clf.predict_proba(X_val)[:, 1]
        y_pred = (proba >= 0.5).astype(int)
        return balanced_accuracy_score(y_val, y_pred)
    return objective

pso_studies = {}   # reuse study.best_params later
optuna_results = {}

for name in MODELS:
    print(f"\n{'='*50}")
    print(f"Optuna HPO — {name}")
    study = optuna.create_study(direction="maximize")
    study.optimize(get_objective(name), n_trials=N_TRIALS, show_progress_bar=True)
    pso_studies[name] = study
    print(f"  Best Balanced Acc: {study.best_value:.4f}")
    print(f"  Best Params: {study.best_params}")

print("\nAll Optuna studies done.")


Optuna HPO — Decision Tree


  0%|          | 0/30 [00:00<?, ?it/s]

  Best Balanced Acc: 0.9726
  Best Params: {'max_depth': 29, 'min_samples_split': 9, 'min_samples_leaf': 7}

Optuna HPO — Random Forest


  0%|          | 0/30 [00:00<?, ?it/s]

  Best Balanced Acc: 0.9724
  Best Params: {'n_estimators': 153, 'max_depth': 30, 'min_samples_split': 4}

Optuna HPO — XGBoost


  0%|          | 0/30 [00:00<?, ?it/s]

  Best Balanced Acc: 0.9717
  Best Params: {'n_estimators': 363, 'max_depth': 10, 'learning_rate': 0.09518652633213662, 'subsample': 0.9179347853983829, 'colsample_bytree': 0.873667245116082}

Optuna HPO — KNN


  0%|          | 0/30 [00:00<?, ?it/s]

In [5]:
import numpy as np
import pandas as pd
import optuna
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── make sure pso_studies exists (Cell B may not have run) ────────────────────
if "pso_studies" not in dir():
    pso_studies = {}

TRAIN_SAMPLE = 50_000
VAL_SAMPLE   = 10_000
N_TRIALS     = 30

def _subsample(X, y, n):
    sss = StratifiedShuffleSplit(n_splits=1, train_size=n, random_state=RANDOM_STATE)
    idx, _ = next(sss.split(X, y))
    X_s = X.iloc[idx] if isinstance(X, pd.DataFrame) else X[idx]
    y_s = y.iloc[idx] if isinstance(y, pd.Series)    else y[idx]
    return X_s, y_s

X_sub,  y_sub  = _subsample(X_train, y_train, TRAIN_SAMPLE)
X_vsub, y_vsub = _subsample(X_val,   y_val,   VAL_SAMPLE)

print(f"Train subsample : {len(X_sub):,}  |  {np.bincount(y_sub.values.astype(int))}")
print(f"Val   subsample : {len(X_vsub):,}  |  {np.bincount(y_vsub.values.astype(int))}")

def knn_objective(trial):
    clf = KNeighborsClassifier(
        n_neighbors = trial.suggest_int("n_neighbors", 3, 20),
        weights     = trial.suggest_categorical("weights", ["uniform", "distance"]),
        p           = trial.suggest_int("p", 1, 2),
        algorithm   = "ball_tree",
        n_jobs      = -1
    )
    clf.fit(X_sub, y_sub)
    y_pred = clf.predict(X_vsub)
    return balanced_accuracy_score(y_vsub, y_pred)

knn_study = optuna.create_study(
    direction = "maximize",
    sampler   = optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner    = optuna.pruners.MedianPruner(n_startup_trials=5),
)
knn_study.optimize(knn_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest Balanced Acc : {knn_study.best_value:.4f}")
print(f"Best Params       : {knn_study.best_params}")

best_knn = KNeighborsClassifier(**knn_study.best_params, n_jobs=-1)
best_knn.fit(X_train, y_train)
pso_studies["KNN"] = knn_study

Train subsample : 50,000  |  [41535  8465]
Val   subsample : 10,000  |  [8307 1693]


  0%|          | 0/30 [00:00<?, ?it/s]


Best Balanced Acc : 0.9712
Best Params       : {'n_neighbors': 5, 'weights': 'uniform', 'p': 2}


In [2]:
# Cell 0b — Load splits
import pickle

with open("models/splits.pkl", "rb") as f:
    splits = pickle.load(f)

X_train = splits["X_train"]
X_val   = splits["X_val"]
X_test  = splits["X_test"]
y_train = splits["y_train"]
y_val   = splits["y_val"]
y_test  = splits["y_test"]

print(f"Train : {X_train.shape}")
print(f"Val   : {X_val.shape}")
print(f"Test  : {X_test.shape}")

Train : (11363049, 34)
Val   : (2434939, 34)
Test  : (2434940, 34)


In [3]:
# Cell 0c — Load baseline results from notebook 02
import os

if os.path.exists("models/ml/all_results.pkl"):
    with open("models/ml/all_results.pkl", "rb") as f:
        all_results = pickle.load(f)
    print(f"Loaded {len(all_results)} baseline results")
else:
    all_results = {}
    print("Warning: no baseline results found — summary table will show '—' for baseline column")

Loaded 7 baseline results


In [5]:
# Cell DL — Optuna HPO for PyTorch MLP
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import optuna
import time
from sklearn.metrics import balanced_accuracy_score
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Config ────────────────────────────────────────────────────────────
DL_SAMPLE  = 200_000   # rows sampled per trial for speed
DL_TRIALS  = 20        # Optuna trials (each trains one MLP config)
DL_EPOCHS  = 10        # max epochs per trial
PATIENCE   = 2         # early stopping inside each trial
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {DEVICE}')
print(f'Sample  : {DL_SAMPLE:,} rows per trial')
print(f'Trials  : {DL_TRIALS}')

# ── Helper: safe iloc-based sampling (fixes KeyError on non-reset index) ──
def to_numpy(arr, pos_idx):
    """Extract rows by positional index regardless of DataFrame/ndarray type."""
    if hasattr(arr, 'iloc'):
        return arr.iloc[pos_idx].values.astype(np.float32)
    return np.asarray(arr)[pos_idx].astype(np.float32)

# ── Pre-convert full val set once (it's fixed across all trials) ──────
X_vl_np = to_numpy(X_val, np.arange(len(X_val)))
y_vl_np = (y_val.values if hasattr(y_val, 'values') else np.asarray(y_val)).astype(np.float32)
val_ds  = TensorDataset(torch.from_numpy(X_vl_np), torch.from_numpy(y_vl_np))
val_dl  = DataLoader(val_ds, batch_size=4096, shuffle=False, num_workers=0)

n_features = X_vl_np.shape[1]

# ── Dynamic MLP: Optuna picks depth, width, dropout, lr, batch_size ──
class DynamicMLP(nn.Module):
    def __init__(self, in_dim, hidden_size, n_layers, dropout):
        super().__init__()
        layers = []
        prev = in_dim
        for _ in range(n_layers):
            layers += [nn.Linear(prev, hidden_size), nn.BatchNorm1d(hidden_size), nn.ReLU(), nn.Dropout(dropout)]
            prev = hidden_size
            hidden_size = max(32, hidden_size // 2)   # halve each layer
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(1)

# ── Optuna objective ──────────────────────────────────────────────────
def dl_objective(trial):
    # — hyperparameters to search —
    hidden_size = trial.suggest_categorical('hidden_size', [128, 256, 512])
    n_layers    = trial.suggest_int('n_layers', 2, 4)
    dropout     = trial.suggest_float('dropout', 0.1, 0.5)
    lr          = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    batch_size  = trial.suggest_categorical('batch_size', [1024, 2048, 4096])
    weight_decay= trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)

    # — fresh sample each trial (positional, avoids KeyError) —
    rng = np.random.default_rng(trial.number)   # different seed per trial
    pos = rng.choice(len(X_train), size=min(DL_SAMPLE, len(X_train)), replace=False)
    X_tr_np = to_numpy(X_train, pos)
    y_tr_np = (y_train.values if hasattr(y_train, 'values') else np.asarray(y_train))[pos].astype(np.float32)

    train_ds = TensorDataset(torch.from_numpy(X_tr_np), torch.from_numpy(y_tr_np))
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)

    model     = DynamicMLP(n_features, hidden_size, n_layers, dropout).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    best_acc, no_improve = 0.0, 0
    for epoch in range(DL_EPOCHS):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for xb, yb in val_dl:
                logits = model(xb.to(DEVICE)).cpu()
                preds.append((torch.sigmoid(logits) >= 0.5).int().numpy())
                trues.append(yb.int().numpy())
        acc = balanced_accuracy_score(np.concatenate(trues), np.concatenate(preds))

        trial.report(acc, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        if acc > best_acc:
            best_acc, no_improve = acc, 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                break

    return best_acc

# ── Run Optuna ────────────────────────────────────────────────────────
print(f'\nRunning {DL_TRIALS} Optuna trials for MLP...')
dl_study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=3)  # kill bad trials early
)
t_start = time.time()
dl_study.optimize(dl_objective, n_trials=DL_TRIALS, show_progress_bar=True)
elapsed = time.time() - t_start

print(f'\nDone in {elapsed/60:.1f} min')
print(f'Best Balanced Acc : {dl_study.best_value:.4f}')
print(f'Best Params       : {dl_study.best_params}')

# ── Retrain best config on full sample ───────────────────────────────
print('\nRetraining best config on full sample...')
bp = dl_study.best_params
rng = np.random.default_rng(RANDOM_STATE)
pos = rng.choice(len(X_train), size=min(DL_SAMPLE, len(X_train)), replace=False)
X_tr_np = to_numpy(X_train, pos)
y_tr_np = (y_train.values if hasattr(y_train, 'values') else np.asarray(y_train))[pos].astype(np.float32)

train_ds = TensorDataset(torch.from_numpy(X_tr_np), torch.from_numpy(y_tr_np))
train_dl = DataLoader(train_ds, batch_size=bp['batch_size'], shuffle=True, num_workers=0)

best_model = DynamicMLP(n_features, bp['hidden_size'], bp['n_layers'], bp['dropout']).to(DEVICE)
optimizer  = torch.optim.Adam(best_model.parameters(), lr=bp['lr'], weight_decay=bp['weight_decay'])
criterion  = nn.BCEWithLogitsLoss()

best_acc, best_state, no_improve = 0.0, None, 0
print(f"{'Epoch':<7} {'Val Bal-Acc':<14} {'Time':>6}")
print('-' * 30)
for epoch in range(1, 20):
    t0 = time.time()
    best_model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        criterion(best_model(xb), yb).backward()
        optimizer.step()
    best_model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in val_dl:
            logits = best_model(xb.to(DEVICE)).cpu()
            preds.append((torch.sigmoid(logits) >= 0.5).int().numpy())
            trues.append(yb.int().numpy())
    acc = balanced_accuracy_score(np.concatenate(trues), np.concatenate(preds))
    marker = ' ◀ best' if acc > best_acc else ''
    print(f"{epoch:<7} {acc:<14.4f} {time.time()-t0:>5.1f}s{marker}")
    if acc > best_acc:
        best_acc  = acc
        best_state = {k: v.cpu().clone() for k, v in best_model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= 3:
            print('Early stopping.')
            break

best_model.load_state_dict(best_state)
print(f'\nFinal Val Balanced Acc: {best_acc:.4f}')

# ── Store result ──────────────────────────────────────────────────────
if 'all_results' not in dir():
    all_results = {}
all_results['MLP (DL)'] = {
    'balanced_accuracy': best_acc,
    'model': best_model,
    'best_params': bp,
    'study': dl_study,
    'framework': 'PyTorch'
}
print("Stored in all_results['MLP (DL)']")
# torch.save(best_state, 'models/ml/mlp_best.pt')  # optional


Device  : cuda
Sample  : 200,000 rows per trial
Trials  : 20

Running 20 Optuna trials for MLP...


  0%|          | 0/20 [00:00<?, ?it/s]


Done in 23.6 min
Best Balanced Acc : 0.9700
Best Params       : {'hidden_size': 128, 'n_layers': 2, 'dropout': 0.12510749961346435, 'lr': 0.000615297673966871, 'batch_size': 1024, 'weight_decay': 0.0001509558606132874}

Retraining best config on full sample...
Epoch   Val Bal-Acc      Time
------------------------------
1       0.9551          12.2s ◀ best
2       0.9611          12.2s ◀ best
3       0.9651          12.7s ◀ best
4       0.9636          13.0s
5       0.9643          12.2s
6       0.9688          12.1s ◀ best
7       0.9677          12.3s
8       0.9689          12.8s ◀ best
9       0.9691          12.9s ◀ best
10      0.9692          13.0s ◀ best
11      0.9690          12.5s
12      0.9689          12.9s
13      0.9692          13.0s ◀ best
14      0.9692          12.9s
15      0.9695          13.4s ◀ best
16      0.9696          12.6s ◀ best
17      0.9692          13.1s
18      0.9694          13.2s
19      0.9698          13.0s ◀ best

Final Val Balanced Acc: 0.969